# LSTM Training For Colab

This notebook mirrors `src/lstm/train_lstm_torch.py`, keeps the same feature engineering and payload structure, and writes a pipeline-compatible artifact into `pipeline/models`.

Estimated training time on the current dataset:
- Colab T4 GPU with mixed precision, full dataset, `5` CV folds + final fit: about `45-110 minutes`
- Colab CPU only: about `3-6 hours`
- `max_tickers=200`: often `10-25 minutes`


In [ ]:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU
!pip -q install prophet xgboost torch scikit-learn pandas numpy joblib


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/diploma')  # Change if your repo is elsewhere in Colab or Drive.
DATASET_PATH = PROJECT_DIR / 'dataset' / 'stock_prices_20y.db'
MODELS_DIR = PROJECT_DIR / 'pipeline' / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for path in [MODELS_DIR, RESULTS_DIR, ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset at {DATASET_PATH}. Update PROJECT_DIR or place the repo there first.'
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project dir :', PROJECT_DIR)
print('Dataset path :', DATASET_PATH)


In [ ]:
import os
import sqlite3
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_ENABLED = DEVICE.type == 'cuda'
print('Torch device:', DEVICE)
if GPU_ENABLED:
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
else:
    print('No GPU detected. The notebook still runs, just slower.')

with sqlite3.connect(DATASET_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')


In [ ]:
from src.lstm.train_lstm_torch import (
    Config,
    LSTMModel,
    SequenceDataset,
    add_target,
    build_grouped_time_folds,
    build_sequences,
    fit_scaler,
    load_prices_from_sqlite,
    make_sequence_features,
    regression_metrics,
    set_seed,
    split_train_test_per_ticker,
)

RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

cfg = Config(
    db_path=str(DATASET_PATH),
    model_output_path=str(MODELS_DIR / f'{RUN_TAG}-lstm_torch.pkl'),
    config_output_path=str(ARTIFACTS_DIR / f'{RUN_TAG}-lstm_torch_config.json'),
)
if GPU_ENABLED:
    cfg.batch_size = 4096
print(cfg)


In [ ]:
import gc
import json
import pickle
from dataclasses import asdict

import torch
from torch.utils.data import DataLoader

NUM_WORKERS = min(2, os.cpu_count() or 0)
AMP_ENABLED = DEVICE.type == 'cuda'


def make_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        persistent_workers=NUM_WORKERS > 0,
    )


def train_model_amp(model, train_loader, val_loader, cfg):
    criterion = torch.nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs, eta_min=cfg.learning_rate * 0.05
    )
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss, n_train = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
            yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                pred = model(xb)
                loss = criterion(pred, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(yb)
            n_train += len(yb)

        scheduler.step()
        train_loss = total_loss / max(n_train, 1)

        model.eval()
        total_val, n_val = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
                yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                    pred = model(xb)
                    val_loss = criterion(pred, yb)
                total_val += val_loss.item() * len(yb)
                n_val += len(yb)
        val_loss = total_val / max(n_val, 1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f'Epoch {epoch:3d} | train={train_loss:.6f} | val={val_loss:.6f} | '
                f'lr={scheduler.get_last_lr()[0]:.2e}',
                flush=True,
            )
        if patience_counter >= cfg.patience:
            print(f'Early stop at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    preds = []
    for xb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
            preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)


In [ ]:
start_time = time.perf_counter()
set_seed(cfg.random_state)

print('Loading data...')
raw_df = load_prices_from_sqlite(cfg.db_path, cfg.table_name)
all_tickers = list(raw_df['ticker'].unique())
if cfg.max_tickers > 0 and len(all_tickers) > cfg.max_tickers:
    rng = np.random.default_rng(cfg.random_state)
    all_tickers = rng.choice(all_tickers, cfg.max_tickers, replace=False).tolist()
    raw_df = raw_df[raw_df['ticker'].isin(all_tickers)].reset_index(drop=True)
    print(f'Subsampled to {cfg.max_tickers} tickers')
else:
    print(f'Using all {len(all_tickers)} tickers')

feat_df = make_sequence_features(raw_df)
train_feat_df, test_feat_df = split_train_test_per_ticker(feat_df, cfg.test_size, cfg.target_horizon)
train_df = add_target(train_feat_df, cfg.target_horizon, cfg.predict_target)
test_df = add_target(test_feat_df, cfg.target_horizon, cfg.predict_target)

X_train_raw, y_train, meta_train = build_sequences(train_df, list(cfg.feature_cols), cfg.sequence_length)
X_test_raw, y_test, meta_test = build_sequences(test_df, list(cfg.feature_cols), cfg.sequence_length)
print(f'Train sequences: {len(X_train_raw):,} | Test sequences: {len(X_test_raw):,}')

folds = build_grouped_time_folds(meta_train, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

n_features = len(cfg.feature_cols)
cv_results = []
for fold_i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    scaler = fit_scaler(X_train_raw, tr_idx)
    train_ds = SequenceDataset(X_train_raw, y_train, tr_idx, scaler)
    val_ds = SequenceDataset(X_train_raw, y_train, va_idx, scaler)
    train_loader = make_loader(train_ds, cfg.batch_size, True)
    val_loader = make_loader(val_ds, cfg.batch_size, False)

    model = LSTMModel(n_features, cfg).to(DEVICE)
    model = train_model_amp(model, train_loader, val_loader, cfg)
    metrics = regression_metrics(y_train[va_idx], predict_model(model, val_loader))
    cv_results.append(metrics)
    print(metrics)

    del model, scaler, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if GPU_ENABLED:
        torch.cuda.empty_cache()

cv_df = pd.DataFrame(cv_results)
print('\nCV mean metrics')
print(cv_df.mean(numeric_only=True).to_string())

all_idx = np.arange(len(X_train_raw), dtype=np.int64)
scaler = fit_scaler(X_train_raw, all_idx)
split = int(len(all_idx) * 0.9)
train_ds = SequenceDataset(X_train_raw, y_train, all_idx[:split], scaler)
val_ds = SequenceDataset(X_train_raw, y_train, all_idx[split:], scaler)
train_loader = make_loader(train_ds, cfg.batch_size, True)
val_loader = make_loader(val_ds, cfg.batch_size, False)

final_model = LSTMModel(n_features, cfg).to(DEVICE)
final_model = train_model_amp(final_model, train_loader, val_loader, cfg)

test_idx = np.arange(len(X_test_raw), dtype=np.int64)
test_ds = SequenceDataset(X_test_raw, y_test, test_idx, scaler)
test_loader = make_loader(test_ds, cfg.batch_size, False)
y_test_pred = predict_model(final_model, test_loader)
test_metrics = regression_metrics(y_test, y_test_pred)
print('Held-out test metrics:', test_metrics)

cpu_state_dict = {k: v.detach().cpu() for k, v in final_model.state_dict().items()}
payload = {
    'model_state_dict': cpu_state_dict,
    'scaler': scaler,
    'config': asdict(cfg),
    'cv_results': cv_results,
    'test_metrics': test_metrics,
}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
config_dict = asdict(cfg)
config_dict['feature_cols'] = list(config_dict['feature_cols'])
with open(cfg.config_output_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-lstm_torch_predictions.csv'
out = meta_test.copy()
out['y_true'] = y_test
out['y_pred'] = y_test_pred
out.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
